# Quantum Oracles, Phase Kickback & the Deutsch–Jozsa Algorithm
### A Step-by-Step Qiskit 1.x Demonstration for Google Colab

---

## Overview

This notebook takes you on a guided tour of three deeply connected ideas in quantum computing:

| Concept | What it means |
|---|---|
| **Quantum Oracle** | A black-box unitary that encodes a classical function $f(x)$ into quantum phase or amplitude |
| **Phase Kickback** | A trick where a controlled-U gate 'kicks' the phase of U's eigenvalue back onto the *control* qubit |
| **Deutsch–Jozsa Algorithm** | The first quantum algorithm that exponentially outperforms any classical algorithm for a specific problem |

### Problem Statement (Deutsch–Jozsa)
Given a black-box function $f:\{0,1\}^n \to \{0,1\}$, we are **promised** that $f$ is either:
- **Constant** — outputs 0 for all inputs, or 1 for all inputs
- **Balanced** — outputs 0 for exactly half of all inputs and 1 for the other half

**Classical cost:** Up to $2^{n-1}+1$ queries in the worst case.  
**Quantum cost:** Exactly **1 query** — always!

---

**Structure of this notebook:**
1. Installation & Imports
2. What is a Quantum Oracle?
3. Phase Kickback — Theory & Demo
4. Deutsch–Jozsa for 1 qubit (original Deutsch's problem)
5. Deutsch–Jozsa for n qubits — Constant Oracle
6. Deutsch–Jozsa for n qubits — Balanced Oracle
7. Full Visualization & Results Summary
8. Interactive: Test Any Oracle You Like!

---
## Cell 1 — Installation

We install:
- `qiskit` — the core quantum SDK (version ≥ 1.0)
- `qiskit-aer` — high-performance quantum circuit simulator
- `pylatexenc` — needed for circuit drawing in Matplotlib mode

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Package Installation                              ║
# ╚══════════════════════════════════════════════════════════════╝
# Run this cell first. It will take ~60 seconds on a fresh Colab runtime.

!pip install qiskit>=1.0 qiskit-aer pylatexenc --quiet
print(" Installation complete.")

---
## Cell 2 — Imports & Version Check

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Imports & Version Check                           ║
# ╚══════════════════════════════════════════════════════════════╝

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display

# --- Qiskit core ---
import qiskit
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector, partial_trace, DensityMatrix
from qiskit.visualization import plot_histogram, plot_bloch_multivector

# --- Qiskit Aer simulator ---
from qiskit_aer import AerSimulator

# --- Plot settings ---
matplotlib.rcParams.update({
    'figure.facecolor': '#0f0f23',
    'axes.facecolor':   '#1a1a2e',
    'axes.edgecolor':   '#888',
    'axes.labelcolor':  '#ddd',
    'xtick.color':      '#ccc',
    'ytick.color':      '#ccc',
    'text.color':       '#eee',
    'grid.color':       '#333',
    'font.family':      'monospace',
})

SIMULATOR = AerSimulator()
SHOTS     = 4096

print(f"✅ Qiskit version : {qiskit.__version__}")
print(f"✅ Simulator      : {SIMULATOR.name}")
print(f"✅ Shots per run  : {SHOTS}")

---
## Cell 3 — What Is a Quantum Oracle?

A **quantum oracle** is a unitary operator $U_f$ that implements a classical Boolean function $f(x)$ reversibly:

$$U_f \,|x\rangle|y\rangle = |x\rangle|y \oplus f(x)\rangle$$

where:
- $|x\rangle$ is the **query register** (n qubits) — holds the input
- $|y\rangle$ is the **target/ancilla register** (1 qubit) — receives $f(x)$ XOR-ed in
- $\oplus$ is bitwise XOR (classical addition mod 2)

### Why reversible?
Quantum mechanics requires all operations to be **unitary** (reversible). A classical function $f$ may not be reversible (many-to-one), but $U_f$ always is — you can recover $|x\rangle$ from the output because $x$ is never destroyed.

### Two oracle types for Deutsch–Jozsa:

| Oracle type | What it does | Circuit implementation |
|---|---|---|
| **Constant-0** | $f(x)=0 \,\forall x$ | Identity (do nothing) |
| **Constant-1** | $f(x)=1 \,\forall x$ | X gate on target |
| **Balanced** | $f(x)=0$ for half, $1$ for other half | CNOT(s) from query to target |

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Oracle Demonstration (1 query qubit)              ║
# ╚══════════════════════════════════════════════════════════════╝

def make_oracle_constant0(n: int) -> QuantumCircuit:
    """
    Constant-0 oracle: f(x) = 0 for ALL x.
    U_f|x>|y> = |x>|y XOR 0> = |x>|y>  (identity — no gates needed)

    Parameters
    ----------
    n : int  — number of query qubits

    Returns
    -------
    QuantumCircuit with n+1 qubits (n query + 1 ancilla)
    """
    qc = QuantumCircuit(n + 1, name="Oracle_const0")
    # Nothing to do: XOR with 0 is identity
    return qc


def make_oracle_constant1(n: int) -> QuantumCircuit:
    """
    Constant-1 oracle: f(x) = 1 for ALL x.
    U_f|x>|y> = |x>|y XOR 1> = |x>|NOT y>
    Implemented by a single X (Pauli-X / NOT) gate on the ancilla.

    Parameters
    ----------
    n : int  — number of query qubits
    """
    qc = QuantumCircuit(n + 1, name="Oracle_const1")
    qc.x(n)   # flip ancilla qubit (index n) unconditionally
    return qc


def make_oracle_balanced(n: int) -> QuantumCircuit:
    """
    Balanced oracle: f(x) = x_0 XOR x_1 XOR ... XOR x_{n-1}.
    This is balanced because the XOR of n bits equals 0 for exactly
    half of all 2^n bitstrings and 1 for the other half.

    Implementation: CNOT from every query qubit to the ancilla.
    U_f|x>|y> = |x>|y XOR x_0 XOR x_1 XOR ... XOR x_{n-1}>

    Parameters
    ----------
    n : int  — number of query qubits
    """
    qc = QuantumCircuit(n + 1, name="Oracle_balanced")
    for qubit in range(n):
        qc.cx(qubit, n)   # CNOT: control=query[i], target=ancilla
    return qc


# ── Visualise all three oracles for n=2 query qubits ──────────────────────────
n_demo = 2

oracles = [
    (make_oracle_constant0(n_demo), "Constant-0 Oracle\n f(x)=0 always",  "#2ecc71"),
    (make_oracle_constant1(n_demo), "Constant-1 Oracle\n f(x)=1 always",  "#e74c3c"),
    (make_oracle_balanced(n_demo),  "Balanced Oracle\n f(x)=x₀⊕x₁",      "#3498db"),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 3))
fig.suptitle("Quantum Oracle Circuits (n=2 query qubits + 1 ancilla)",
             fontsize=14, fontweight='bold', color='white', y=1.05)

for ax, (qc, title, colour) in zip(axes, oracles):
    ax.set_title(title, color=colour, fontsize=10, fontweight='bold')
    circuit_fig = qc.draw(output='mpl', style='iqp', fold=-1)
    # Render to canvas buffer then show in subplot
    circuit_fig.savefig('/tmp/_oracle_tmp.png', bbox_inches='tight',
                        facecolor='#1a1a2e', dpi=110)
    plt.close(circuit_fig)
    img = plt.imread('/tmp/_oracle_tmp.png')
    ax.imshow(img)
    ax.axis('off')

plt.tight_layout()
plt.savefig('/tmp/oracles.png', bbox_inches='tight', dpi=130, facecolor='#0f0f23')
plt.show()
print("\n📌 Key:  q0, q1 = query qubits   |   q2 = ancilla (target) qubit")

---
## Cell 4 — Phase Kickback: Theory

Phase kickback is the secret engine behind almost every quantum speedup. Here is how it works:

### Setup
Suppose gate $U$ has an eigenstate $|u\rangle$ with eigenvalue $e^{i\theta}$:
$$U|u\rangle = e^{i\theta}|u\rangle$$

Now apply **controlled-U** with a control qubit in superposition $\frac{|0\rangle+|1\rangle}{\sqrt{2}}$:

$$\text{CU}\left(\frac{|0\rangle+|1\rangle}{\sqrt{2}}\right)|u\rangle
= \frac{|0\rangle U^0|u\rangle + |1\rangle U^1|u\rangle}{\sqrt{2}}
= \frac{|0\rangle + e^{i\theta}|1\rangle}{\sqrt{2}} \otimes |u\rangle$$

The phase $e^{i\theta}$ has **kicked back** onto the *control* qubit — even though $U$ acts on the *target*!

### In Deutsch–Jozsa specifically
We set the ancilla to $|{-}\rangle = \frac{|0\rangle - |1\rangle}{\sqrt{2}}$ (an eigenstate of X with eigenvalue $-1$).  
Then:
$$U_f|x\rangle|{-}\rangle = (-1)^{f(x)}|x\rangle|{-}\rangle$$

The oracle now **marks** query states with a phase of $(-1)^{f(x)}$ — the ancilla is unchanged and can be ignored!

This is the **phase oracle** view of $U_f$.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Phase Kickback Circuit Demo                       ║
# ╚══════════════════════════════════════════════════════════════╝
#
# We demonstrate phase kickback with the simplest possible example:
#   - Control qubit q0 prepared in |+> = H|0>
#   - Target qubit  q1 prepared in |-> = HX|0>  (eigenstate of X, eigenvalue -1)
#   - Gate: CNOT (controlled-X)
#
# Without kickback: q0 should stay |+>.
# With kickback:    q0 becomes  |-> because the -1 phase of |-> kicks back.
#
# We verify by measuring q0 in the X-basis (apply H then measure).
#   |+> in X-basis → always 0
#   |-> in X-basis → always 1   ← this is what we expect to see!

# ── Build the circuit ─────────────────────────────────────────
qc_kick = QuantumCircuit(2, 1, name="Phase Kickback Demo")

# Prepare control qubit q0 in |+>
qc_kick.h(0)

# Prepare target qubit q1 in |-> = X then H
qc_kick.x(1)
qc_kick.h(1)

qc_kick.barrier(label="initial state")

# Apply CNOT: the -1 phase of |-> kicks back to q0
qc_kick.cx(0, 1)           # CNOT(control=q0, target=q1)

qc_kick.barrier(label="after oracle")

# Measure q0 in the X-basis: apply H then measure
# |+> → H → |0>  measures 0
# |-> → H → |1>  measures 1
qc_kick.h(0)
qc_kick.measure(0, 0)

# ── Simulate ──────────────────────────────────────────────────
compiled = transpile(qc_kick, SIMULATOR)
result   = SIMULATOR.run(compiled, shots=SHOTS).result()
counts   = result.get_counts()

# ── Draw circuit ──────────────────────────────────────────────
print("Phase Kickback Circuit:")
display(qc_kick.draw(output='mpl', style='iqp', fold=-1))

# ── Plot measurement result ────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Phase Kickback Demonstration", fontsize=14,
             fontweight='bold', color='white')

# Bloch sphere — show state of q0 BEFORE and AFTER kickback (no measurement)
# Before: state after H on q0 only
qc_before = QuantumCircuit(1)
qc_before.h(0)
sv_before = Statevector(qc_before)

# After:  q0 has the kicked-back phase → effectively |->
qc_after  = QuantumCircuit(1)
qc_after.h(0)
qc_after.z(0)        # Z = phase flip; |+> → |-> (simulates kickback result)
sv_after  = Statevector(qc_after)

# Bar chart of measurement counts
labels = list(counts.keys())
values = list(counts.values())
colours = ['#e74c3c' if k == '1' else '#2ecc71' for k in labels]
ax1.bar(labels, values, color=colours, edgecolor='white', linewidth=0.5)
ax1.set_title("Measurement of q0 in X-basis\n(expect '1' = kickback happened)",
              color='white')
ax1.set_xlabel("Outcome", color='#ccc')
ax1.set_ylabel("Counts", color='#ccc')
for bar, val in zip(ax1.patches, values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             f'{val}', ha='center', color='white', fontsize=12)
ax1.set_ylim(0, SHOTS * 1.15)

# Explanation panel
explanation = (
    "Phase Kickback Mechanism\n"
    "━━━━━━━━━━━━━━━━━━━━━━\n\n"
    "q1 is prepared in |−⟩, the −1\n"
    "eigenstate of the X (CNOT target).\n\n"
    "CNOT applies X on q1, giving:\n"
    "  X|−⟩ = −1·|−⟩\n\n"
    "The phase (−1) is multiplicative:\n"
    "it 'kicks back' onto q0, turning\n"
    "  |+⟩ = (|0⟩+|1⟩)/√2\n"
    "into\n"
    "  |−⟩ = (|0⟩−|1⟩)/√2\n\n"
    "Measuring in X-basis (H then\n"
    "measure) always gives '1'. ✓"
)
ax2.text(0.05, 0.95, explanation, transform=ax2.transAxes,
         fontsize=10, verticalalignment='top', family='monospace',
         color='#aef', bbox=dict(boxstyle='round', facecolor='#111533',
                                  edgecolor='#4488cc', alpha=0.9))
ax2.axis('off')
ax2.set_title("Conceptual Explanation", color='white')

plt.tight_layout()
plt.savefig('/tmp/kickback.png', bbox_inches='tight', dpi=130, facecolor='#0f0f23')
plt.show()

print(f"\n Counts: {counts}")
print("All shots landed on '1' — phase kickback confirmed!")

---
## Cell 5 — Deutsch's Algorithm (Original, 1 Query Qubit)

The original **Deutsch algorithm** (1985) solves a 1-bit version:

> Given $f:\{0,1\}\to\{0,1\}$, determine if $f$ is **constant** ($f(0)=f(1)$) or **balanced** ($f(0)\neq f(1)$) in a single query.

### Circuit
```
q0: |0⟩ ─── H ─── Oracle ─── H ─── Measure
q1: |0⟩ ─── X ─── H ─── Oracle (ancilla, not measured)
```

### Math walkthrough
| Step | State |
|---|---|
| Initial | $|0\rangle|0\rangle$ |
| After X on q1 | $|0\rangle|1\rangle$ |
| After H on both | $|{+}\rangle|{-}\rangle = \frac{1}{2}(|0\rangle+|1\rangle)(|0\rangle-|1\rangle)$ |
| After oracle (phase kickback) | $\frac{1}{\sqrt{2}}((-1)^{f(0)}|0\rangle + (-1)^{f(1)}|1\rangle)|{-}\rangle$ |
| After H on q0 | **constant** → $|0\rangle$,  **balanced** → $|1\rangle$ |

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Deutsch's Algorithm: all 4 possible functions     ║
# ╚══════════════════════════════════════════════════════════════╝
#
# There are exactly 4 functions f:{0,1}→{0,1}:
#   f0: always 0  (constant)
#   f1: always 1  (constant)
#   f2: identity  f(x)=x     (balanced)
#   f3: negation  f(x)=1-x   (balanced)

def deutsch_circuit(f_type: str) -> QuantumCircuit:
    """
    Build the Deutsch circuit for one of 4 function types.

    Parameters
    ----------
    f_type : 'const0' | 'const1' | 'balanced_id' | 'balanced_neg'

    Returns
    -------
    QuantumCircuit (2 qubits, 1 classical bit)
    """
    qc = QuantumCircuit(2, 1, name=f"Deutsch_{f_type}")

    # Step 1: Prepare ancilla in |1>
    qc.x(1)

    # Step 2: Hadamard on both qubits → |+>|−>
    qc.h(0)
    qc.h(1)

    qc.barrier(label="pre-oracle")

    # Step 3: Apply the oracle
    if f_type == 'const0':
        pass                  # Identity: no gates
    elif f_type == 'const1':
        qc.x(1)              # Flip ancilla: f=1 always
    elif f_type == 'balanced_id':
        qc.cx(0, 1)          # CNOT: f(x)=x
    elif f_type == 'balanced_neg':
        qc.cx(0, 1)          # f(x)=1-x = x XOR 1, so CNOT then X on ancilla
        qc.x(1)

    qc.barrier(label="post-oracle")

    # Step 4: Hadamard on query qubit — interference!
    qc.h(0)

    # Step 5: Measure query qubit
    # |0> → constant, |1> → balanced
    qc.measure(0, 0)

    return qc


functions = {
    'const0':       ('Constant-0\n f(0)=f(1)=0',   '#27ae60', 'CONSTANT'),
    'const1':       ('Constant-1\n f(0)=f(1)=1',   '#2ecc71', 'CONSTANT'),
    'balanced_id':  ('Balanced-ID\n f(x)=x',        '#2980b9', 'BALANCED'),
    'balanced_neg': ('Balanced-NOT\n f(x)=1−x',    '#3498db', 'BALANCED'),
}

results_deutsch = {}

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle("Deutsch Algorithm — All 4 Functions",
             fontsize=15, fontweight='bold', color='white', y=1.01)

for col, (ftype, (label, colour, truth)) in enumerate(functions.items()):
    qc = deutsch_circuit(ftype)

    # ── Draw circuit ──
    circuit_img = qc.draw(output='mpl', style='iqp', fold=-1)
    circuit_img.savefig('/tmp/_deu_tmp.png', bbox_inches='tight',
                        facecolor='#1a1a2e', dpi=100)
    plt.close(circuit_img)
    img = plt.imread('/tmp/_deu_tmp.png')
    axes[0, col].imshow(img)
    axes[0, col].axis('off')
    axes[0, col].set_title(label, color=colour, fontsize=10, fontweight='bold')

    # ── Simulate ──
    compiled = transpile(qc, SIMULATOR)
    counts   = SIMULATOR.run(compiled, shots=SHOTS).result().get_counts()
    results_deutsch[ftype] = (counts, truth)

    measured_0 = counts.get('0', 0)
    measured_1 = counts.get('1', 0)
    predicted  = 'CONSTANT' if measured_0 > measured_1 else 'BALANCED'
    correct    = predicted == truth

    # ── Plot counts ──
    bar_colors = ['#2ecc71' if '0' in counts else '#555',
                  '#e74c3c' if '1' in counts else '#555']
    axes[1, col].bar(['0\n(→constant)', '1\n(→balanced)'],
                     [measured_0, measured_1],
                     color=['#2ecc71', '#e74c3c'],
                     edgecolor='white', linewidth=0.5)
    verdict = f"{predicted}" if correct else f"❌ {predicted}"
    axes[1, col].set_title(f"Result: {verdict}\nTruth: {truth}",
                           color='#aef', fontsize=9)
    axes[1, col].set_ylim(0, SHOTS * 1.2)
    axes[1, col].set_ylabel('Counts')

plt.tight_layout()
plt.savefig('/tmp/deutsch.png', bbox_inches='tight', dpi=130, facecolor='#0f0f23')
plt.show()

print("\n📋 Summary:")
print(f"{'Function':<20} {'Counts':>15}  {'Truth':>10}  {'Algorithm Says'}")
print("─" * 65)
for ftype, (counts, truth) in results_deutsch.items():
    m0 = counts.get('0', 0)
    m1 = counts.get('1', 0)
    predicted = 'CONSTANT' if m0 > m1 else 'BALANCED'
    ok = 'ok' if predicted == truth else 'Not Ok'
    print(f"{ftype:<20}  0:{m0:>4}  1:{m1:>4}   {truth:>10}  {ok} {predicted}")

---
## Cell 6 — Deutsch–Jozsa for n Qubits

The generalised **Deutsch–Jozsa algorithm** solves the same problem for $n$ input bits.

### Full Circuit (n query qubits + 1 ancilla)

```
q0 : |0⟩ ── H ──┐            ┌── H ── Measure
q1 : |0⟩ ── H ──┤  U_f       ├── H ── Measure
...              │  (oracle)  │
q_{n-1}: |0⟩ H─┤            ├── H ── Measure
ancilla: |0⟩ XH─┘            └─────── (ignored)
```

### Mathematical Analysis

After the first $H^{\otimes n}$ on the query register:
$$|q\rangle = \frac{1}{\sqrt{2^n}}\sum_{x=0}^{2^n-1}|x\rangle$$

After the oracle with ancilla in $|{-}\rangle$ (phase kickback):
$$|q\rangle = \frac{1}{\sqrt{2^n}}\sum_{x=0}^{2^n-1}(-1)^{f(x)}|x\rangle$$

After the second $H^{\otimes n}$:
$$|q\rangle = \sum_{z=0}^{2^n-1}\left[\frac{1}{2^n}\sum_{x=0}^{2^n-1}(-1)^{f(x)+x\cdot z}\right]|z\rangle$$

The amplitude of $|0\cdots 0\rangle$ is:
$$\text{amp}(0\cdots0) = \frac{1}{2^n}\sum_{x=0}^{2^n-1}(-1)^{f(x)} = \begin{cases}\pm 1 & f\text{ constant}\\ 0 & f\text{ balanced}\end{cases}$$

**Decision rule:**  Measure all query qubits.
- All zeros `00...0` → **constant**
- Any non-zero result → **balanced**

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 6 — n-qubit Deutsch–Jozsa Circuit Builder             ║
# ╚══════════════════════════════════════════════════════════════╝

def deutsch_jozsa_circuit(n: int, oracle_type: str) -> QuantumCircuit:
    """
    Build the complete Deutsch–Jozsa circuit for n query qubits.

    Parameters
    ----------
    n           : number of query qubits
    oracle_type : 'constant0' | 'constant1' | 'balanced'

    Returns
    -------
    QuantumCircuit (n+1 qubits, n classical bits)

    Circuit layout
    --------------
    q[0..n-1] : query register  (initialised |0>)
    q[n]      : ancilla qubit   (initialised |0>, becomes |−>)
    c[0..n-1] : classical bits for measurement
    """
    query   = QuantumRegister(n,   name='q')     # query qubits
    ancilla = QuantumRegister(1,   name='anc')   # ancilla
    creg    = ClassicalRegister(n, name='c')     # measurement results

    qc = QuantumCircuit(query, ancilla, creg,
                        name=f"DJ_{oracle_type}_n{n}")

    # ── Step 1: Initialise ancilla in |1> ──────────────────────
    qc.x(ancilla[0])

    # ── Step 2: Hadamard on ALL qubits ─────────────────────────
    # Query qubits: |0>^n  → |+>^n  = equal superposition of all inputs
    # Ancilla:      |1>    → |->    = (|0>-|1>)/√2  (phase kickback target)
    qc.h(query)
    qc.h(ancilla[0])

    qc.barrier(label="pre-oracle")

    # ── Step 3: Apply the oracle ────────────────────────────────
    if oracle_type == 'constant0':
        # f(x)=0 for all x → identity, no gates
        pass

    elif oracle_type == 'constant1':
        # f(x)=1 for all x → X on ancilla flips every amplitude's sign
        qc.x(ancilla[0])

    elif oracle_type == 'balanced':
        # f(x) = x_0 XOR x_1 XOR ... XOR x_{n-1}
        # Implemented as CNOT from each query qubit to ancilla
        for i in range(n):
            qc.cx(query[i], ancilla[0])

    else:
        raise ValueError(f"Unknown oracle_type: {oracle_type}")

    qc.barrier(label="post-oracle")

    # ── Step 4: Second Hadamard on query qubits ─────────────────
    # This creates interference: amplitudes of |0...0> add up
    # constructively (constant) or cancel to 0 (balanced)
    qc.h(query)

    # ── Step 5: Measure query qubits ────────────────────────────
    # All zeros → constant;  any non-zero → balanced
    qc.measure(query, creg)

    return qc


# ── Quick sanity check with n=3 ────────────────────────────────
n_test = 3
qc_const    = deutsch_jozsa_circuit(n_test, 'constant1')
qc_balanced = deutsch_jozsa_circuit(n_test, 'balanced')

print("=" * 55)
print(f" Deutsch–Jozsa Circuit — n={n_test} query qubits")
print("=" * 55)

print("\n── Constant-1 Oracle ──")
display(qc_const.draw(output='mpl', style='iqp', fold=20))

print("\n── Balanced Oracle ──")
display(qc_balanced.draw(output='mpl', style='iqp', fold=20))

---
## Cell 7 — Simulation & Comprehensive Visualization

We run the algorithm for multiple values of $n$ and both oracle types,
and produce a complete visual summary.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Full Simulation & Visualization                   ║
# ╚══════════════════════════════════════════════════════════════╝

def run_dj(n: int, oracle_type: str, shots: int = SHOTS) -> dict:
    """
    Run Deutsch–Jozsa and return measurement counts.

    Returns dict mapping bitstring → count, e.g. {'000': 4096}
    """
    qc       = deutsch_jozsa_circuit(n, oracle_type)
    compiled = transpile(qc, SIMULATOR)
    result   = SIMULATOR.run(compiled, shots=shots).result()
    return result.get_counts()


def verdict(counts: dict, n: int) -> str:
    """Determine algorithm's answer from measurement counts."""
    all_zeros = '0' * n
    # If all shots measured '00...0', oracle is constant
    if counts.get(all_zeros, 0) == sum(counts.values()):
        return 'CONSTANT'
    else:
        return 'BALANCED'


# ── Run for n = 1, 2, 3, 4 with both oracle types ─────────────
n_values      = [1, 2, 3, 4]
oracle_types  = ['constant0', 'constant1', 'balanced']
oracle_truths = {'constant0': 'CONSTANT', 'constant1': 'CONSTANT', 'balanced': 'BALANCED'}
oracle_labels = {'constant0': 'Constant-0', 'constant1': 'Constant-1', 'balanced': 'Balanced'}
oracle_colors = {'constant0': '#27ae60', 'constant1': '#2ecc71', 'balanced': '#3498db'}

all_results = {}   # {(n, oracle_type): counts}
print("Running simulations...")
for n in n_values:
    for otype in oracle_types:
        counts = run_dj(n, otype)
        all_results[(n, otype)] = counts
        v = verdict(counts, n)
        ok = '✅' if v == oracle_truths[otype] else '❌'
        print(f"  n={n}  {oracle_labels[otype]:<14}  →  {v:<10}  {ok}")

print("\nAll simulations complete. Generating visualizations...")

# ══════════════════════════════════════════════════════════════
# FIGURE 1: Measurement histograms for n=3
# ══════════════════════════════════════════════════════════════
n_show = 3
fig1, axes1 = plt.subplots(1, 3, figsize=(18, 5))
fig1.suptitle(f"Deutsch–Jozsa Measurement Histograms  (n={n_show})",
              fontsize=14, fontweight='bold', color='white')

for ax, otype in zip(axes1, oracle_types):
    counts = all_results[(n_show, otype)]
    v      = verdict(counts, n_show)
    truth  = oracle_truths[otype]
    ok     = '✅' if v == truth else '❌'
    colour = oracle_colors[otype]

    # Highlight the all-zeros outcome distinctly
    all_zeros = '0' * n_show
    bar_cols  = [('#f0e642' if k == all_zeros else '#e74c3c')
                 for k in counts]

    fig_hist = plot_histogram(
        counts,
        ax=ax,
        color=bar_cols,
        title=(
            f"{oracle_labels[otype]} Oracle\n"
            f"Result: {ok} {v}  |  Truth: {truth}"
        ),
    )
    ax.title.set_color(colour)
    ax.title.set_fontsize(11)

plt.tight_layout()
plt.savefig('/tmp/dj_histograms.png', bbox_inches='tight',
            dpi=130, facecolor='#0f0f23')
plt.show()

print("\n Yellow bar = '00...0' outcome  →  CONSTANT oracle")
print(" Red bars   = non-zero outcomes →  BALANCED oracle")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7b — Statevector Probability Amplitudes               ║
# ╚══════════════════════════════════════════════════════════════╝
#
# We use Statevector simulation (no measurement) to see the exact
# quantum state probabilities BEFORE measurement for n=2.
# This makes the algorithm's magic visible as probability amplitudes.

def dj_no_measure(n: int, oracle_type: str) -> QuantumCircuit:
    """
    Same as deutsch_jozsa_circuit but WITHOUT the measurement step.
    Used for Statevector inspection.
    """
    query   = QuantumRegister(n,   name='q')
    ancilla = QuantumRegister(1,   name='anc')
    qc = QuantumCircuit(query, ancilla, name=f"DJ_sv_{oracle_type}_n{n}")

    qc.x(ancilla[0])
    qc.h(query)
    qc.h(ancilla[0])
    qc.barrier()

    if oracle_type == 'constant0':
        pass
    elif oracle_type == 'constant1':
        qc.x(ancilla[0])
    elif oracle_type == 'balanced':
        for i in range(n):
            qc.cx(query[i], ancilla[0])

    qc.barrier()
    qc.h(query)
    return qc


n_sv = 2   # Use n=2 for readability

fig2, axes2 = plt.subplots(1, 3, figsize=(18, 5))
fig2.suptitle(
    f"Query-register Probability Amplitudes Before Measurement  (n={n_sv})\n"
    "(Ancilla traced out)",
    fontsize=13, fontweight='bold', color='white'
)

for ax, otype in zip(axes2, oracle_types):
    qc_sv = dj_no_measure(n_sv, otype)
    sv    = Statevector(qc_sv)

    # Trace out the ancilla qubit to get query-register probabilities
    # ancilla is qubit index n_sv in our register ordering
    dm = DensityMatrix(sv)
    dm_query = partial_trace(dm, [n_sv])   # trace over ancilla

    probs  = np.real(np.diag(dm_query.data))
    labels = [format(i, f'0{n_sv}b') for i in range(2**n_sv)]

    # Highlight the all-zeros outcome
    bar_cols = ['#f0e642' if lbl == '0'*n_sv else '#e74c3c'
                for lbl in labels]

    bars = ax.bar(labels, probs, color=bar_cols,
                  edgecolor='white', linewidth=0.5)
    for bar, prob in zip(bars, probs):
        if prob > 0.01:
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.01,
                    f'{prob:.2f}', ha='center', color='white', fontsize=10)

    truth = oracle_truths[otype]
    v     = 'CONSTANT' if probs[0] > 0.99 else 'BALANCED'
    ok    = '✅' if v == truth else '❌'

    ax.set_title(f"{oracle_labels[otype]} Oracle\n"
                 f"Result: {ok} {v}",
                 color=oracle_colors[otype], fontsize=11)
    ax.set_xlabel("Basis state of query register", color='#ccc')
    ax.set_ylabel("Probability", color='#ccc')
    ax.set_ylim(0, 1.2)
    ax.axhline(1.0, linestyle='--', color='#555', linewidth=0.8)
    ax.text(0.98, 1.02, '1.0', transform=ax.transAxes,
            color='#888', fontsize=8, ha='right')

plt.tight_layout()
plt.savefig('/tmp/dj_statevector.png', bbox_inches='tight',
            dpi=130, facecolor='#0f0f23')
plt.show()

print("\n🟡 Probability = 1.0 on '00' → CONSTANT (all amplitude concentrated at zero)")
print("🔴 Probability spread across non-zero states → BALANCED")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7c — Quantum vs Classical Query Complexity            ║
# ╚══════════════════════════════════════════════════════════════╝
#
# Classical worst case: 2^(n-1)+1 queries
# Quantum:              always 1 query

ns = np.arange(1, 21)
classical_queries = 2**(ns - 1) + 1
quantum_queries   = np.ones_like(ns)

fig3, (ax_log, ax_lin) = plt.subplots(1, 2, figsize=(14, 5))
fig3.suptitle("Classical vs Quantum Query Complexity",
              fontsize=14, fontweight='bold', color='white')

# Log-scale plot
ax_log.semilogy(ns, classical_queries, 'o-', color='#e74c3c',
                linewidth=2, markersize=5, label='Classical worst case')
ax_log.semilogy(ns, quantum_queries, 's-', color='#2ecc71',
                linewidth=2, markersize=5, label='Quantum (Deutsch–Jozsa)')
ax_log.fill_between(ns, quantum_queries, classical_queries,
                     alpha=0.15, color='#8e44ad', label='Exponential speedup')
ax_log.set_xlabel("Number of input bits n")
ax_log.set_ylabel("Number of oracle queries (log scale)")
ax_log.set_title("Log Scale", color='white')
ax_log.legend(facecolor='#1a1a2e', edgecolor='#666')
ax_log.grid(True, alpha=0.3)

# Annotations for specific n values
for n_ann in [5, 10, 15, 20]:
    c_val = 2**(n_ann - 1) + 1
    ax_log.annotate(f'n={n_ann}\n{c_val:,}',
                    xy=(n_ann, c_val),
                    xytext=(n_ann - 3, c_val * 2),
                    fontsize=7, color='#e74c3c',
                    arrowprops=dict(arrowstyle='->', color='#e74c3c'))

# Circuit depth for n=1 to 20
circuit_depths = []
for n in ns:
    qc_temp = deutsch_jozsa_circuit(n, 'balanced')
    # Remove measurements to get unitary depth
    circuit_depths.append(qc_temp.depth())

ax_lin.plot(ns, circuit_depths, 'd-', color='#3498db',
            linewidth=2, markersize=5, label='DJ circuit depth')
ax_lin.plot(ns, classical_queries, 'o-', color='#e74c3c',
            linewidth=2, markersize=5, label='Classical queries')
ax_lin.set_xlabel("Number of input bits n")
ax_lin.set_ylabel("Cost")
ax_lin.set_title("Linear Scale (small n)", color='white')
ax_lin.set_xlim(1, 10)
ax_lin.set_ylim(0, 300)
ax_lin.legend(facecolor='#1a1a2e', edgecolor='#666')
ax_lin.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/dj_complexity.png', bbox_inches='tight',
            dpi=130, facecolor='#0f0f23')
plt.show()

print("\n📈 Complexity Summary:")
print(f"{'n':>4} | {'Classical (worst)':>20} | {'Quantum':>10} | {'Speedup factor':>15}")
print("─" * 58)
for n in [1, 2, 4, 8, 10, 20]:
    c = 2**(n-1) + 1
    print(f"{n:>4} | {c:>20,} | {1:>10} | {c:>14,}x")

---
## Cell 8 — Bloch Sphere: Visualising Phase Kickback on a Single Qubit

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Bloch Sphere Evolution (n=1 Deutsch case)         ║
# ╚══════════════════════════════════════════════════════════════╝
#
# We trace the QUERY qubit state through all steps of Deutsch's
# algorithm for the balanced oracle (f(x)=x).
#
# Steps:
#   Step 0: |0> (north pole)
#   Step 1: H → |+> (equator, +X)
#   Step 2: After oracle → |-> (equator, -X) — phase kickback!
#   Step 3: H → |1> (south pole) — measured as 1 ← BALANCED confirmed!

steps_balanced = []
labels_balanced = []

# Step 0: |0>
qc0 = QuantumCircuit(1); steps_balanced.append(Statevector(qc0)); labels_balanced.append("Step 0\n|0⟩")

# Step 1: H|0> = |+>
qc1 = QuantumCircuit(1); qc1.h(0); steps_balanced.append(Statevector(qc1)); labels_balanced.append("Step 1\nH|0⟩=|+⟩")

# Step 2: Oracle (balanced, 1 qubit) = CNOT with ancilla |-> → kickback → |−>
qc2 = QuantumCircuit(1); qc2.h(0); qc2.z(0); steps_balanced.append(Statevector(qc2)); labels_balanced.append("Step 2\nOracle → |−⟩")

# Step 3: H|-> = |1>
qc3 = QuantumCircuit(1); qc3.h(0); qc3.z(0); qc3.h(0); steps_balanced.append(Statevector(qc3)); labels_balanced.append("Step 3\nH|−⟩=|1⟩")

fig4 = plot_bloch_multivector(
    Statevector(qc3),     # final state
    title="Final query-qubit state after balanced oracle (should be |1⟩)"
)
fig4.set_facecolor('#0f0f23')
for ax in fig4.axes:
    ax.set_facecolor('#1a1a2e')

plt.savefig('/tmp/bloch_final.png', bbox_inches='tight',
            dpi=130, facecolor='#0f0f23')
plt.show()

# ── Text summary of state evolution ──────────────────────────
print("\n── State evolution for Balanced oracle (f(x)=x, n=1) ──")
print()
for sv, lbl in zip(steps_balanced, labels_balanced):
    vec = sv.data
    amp0 = f"{vec[0].real:+.3f}{vec[0].imag:+.3f}i" if abs(vec[0].imag) > 1e-10 else f"{vec[0].real:+.3f}"
    amp1 = f"{vec[1].real:+.3f}{vec[1].imag:+.3f}i" if abs(vec[1].imag) > 1e-10 else f"{vec[1].real:+.3f}"
    print(f"  {lbl.replace(chr(10),' '):<25} = ({amp0})|0⟩ + ({amp1})|1⟩")

print()
print("🎯 Query qubit ends at |1⟩ → Deutsch says BALANCED ✅")
print()
print("── State evolution for Constant oracle (f(x)=0, n=1) ──")
print()
qc_c0 = QuantumCircuit(1); qc_c0.h(0); qc_c0.h(0)
sv_c0 = Statevector(qc_c0)
vec   = sv_c0.data
print(f"  After H → Oracle(id) → H   = ({vec[0].real:+.3f})|0⟩ + ({vec[1].real:+.3f})|1⟩")
print("🎯 Query qubit ends at |0⟩ → Deutsch says CONSTANT ✅")

---
## Cell 9 — Interactive: Test Any Oracle Configuration!

Change `N_QUBITS` and `ORACLE` below to experiment with different settings.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 9 — Interactive Oracle Tester                         ║
# ╚══════════════════════════════════════════════════════════════╝
# 🔧 CHANGE THESE TWO PARAMETERS:
# ─────────────────────────────────────────────────────────────
N_QUBITS = 4                        # try: 1, 2, 3, 4, 5, ...
ORACLE   = 'balanced'               # try: 'constant0', 'constant1', 'balanced'
# ─────────────────────────────────────────────────────────────

assert ORACLE in ('constant0', 'constant1', 'balanced'), \
    "ORACLE must be 'constant0', 'constant1', or 'balanced'"

# ── Build & run ───────────────────────────────────────────────
qc_interactive = deutsch_jozsa_circuit(N_QUBITS, ORACLE)
compiled        = transpile(qc_interactive, SIMULATOR)
counts          = SIMULATOR.run(compiled, shots=SHOTS).result().get_counts()
v               = verdict(counts, N_QUBITS)
truth           = oracle_truths[ORACLE]
ok              = '✅' if v == truth else '❌'

# ── Draw circuit ──────────────────────────────────────────────
print(f"Circuit for n={N_QUBITS}, Oracle='{ORACLE}':")
display(qc_interactive.draw(output='mpl', style='iqp', fold=25))

# ── Plot histogram ────────────────────────────────────────────
fig5, ax5 = plt.subplots(figsize=(max(8, 2**(N_QUBITS-1)+2), 5))
all_zeros = '0' * N_QUBITS
bar_cols  = ['#f0e642' if k == all_zeros else '#e74c3c' for k in counts]
plot_histogram(counts, ax=ax5, color=bar_cols)
ax5.set_title(
    f"n={N_QUBITS} | Oracle: {oracle_labels[ORACLE]}\n"
    f"Algorithm result: {ok} {v}  |  Ground truth: {truth}",
    color='white', fontsize=12
)
plt.tight_layout()
plt.savefig('/tmp/dj_interactive.png', bbox_inches='tight',
            dpi=130, facecolor='#0f0f23')
plt.show()

# ── Stats ─────────────────────────────────────────────────────
zero_frac   = counts.get(all_zeros, 0) / SHOTS
other_frac  = 1 - zero_frac
gate_count  = qc_interactive.count_ops()

print("\n" + "═" * 55)
print(f"  Deutsch–Jozsa Interactive Results")
print("═" * 55)
print(f"  Input bits (n)       : {N_QUBITS}")
print(f"  Oracle type          : {oracle_labels[ORACLE]}")
print(f"  Shots                : {SHOTS:,}")
print(f"  Unique outcomes      : {len(counts)}")
print(f"  P(all-zeros outcome) : {zero_frac:.4f}")
print(f"  P(non-zero outcomes) : {other_frac:.4f}")
print(f"  Circuit depth        : {qc_interactive.depth()}")
print(f"  Gate counts          : {dict(gate_count)}")
print("─" * 55)
print(f"  Algorithm answer     : {v}")
print(f"  Correct?             : {ok}")
print("═" * 55)
print(f"\n  Classical worst case would have needed "
      f"{2**(N_QUBITS-1)+1:,} queries.")
print(f"  Quantum used: 1 query. Speedup: {2**(N_QUBITS-1)+1:,}x 🚀")

---
## Cell 10 — Summary & Key Takeaways

Run this cell to print a structured recap.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 10 — Summary of Key Takeaways                         ║
# ╚══════════════════════════════════════════════════════════════╝

summary = """
╔══════════════════════════════════════════════════════════════════╗
║     QUANTUM ORACLES · PHASE KICKBACK · DEUTSCH–JOZSA           ║
║                    SUMMARY OF KEY IDEAS                         ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  1. QUANTUM ORACLE  U_f |x⟩|y⟩ = |x⟩|y ⊕ f(x)⟩               ║
║     • Encodes f(x) reversibly into quantum state                ║
║     • Preserves unitarity — required by quantum mechanics       ║
║     • Types for DJ: constant-0, constant-1, balanced            ║
║                                                                  ║
║  2. PHASE KICKBACK                                               ║
║     • Set ancilla to |−⟩ = (|0⟩−|1⟩)/√2                       ║
║     • U_f |x⟩|−⟩ = (−1)^f(x) |x⟩|−⟩                          ║
║     • Phase (−1)^f(x) 'kicks back' to the query register       ║
║     • Ancilla is unchanged → can be discarded                   ║
║                                                                  ║
║  3. DEUTSCH–JOZSA ALGORITHM                                      ║
║     • n query qubits + 1 ancilla qubit                          ║
║     • Steps: X(anc) → H⊗(n+1) → Oracle → H⊗n → Measure        ║
║     • Constant: all query bits measure 0 (probability = 1)      ║
║     • Balanced: at least one non-zero bit (probability = 1)     ║
║     • Classical worst case: 2^(n−1)+1 queries                   ║
║     • Quantum: ALWAYS 1 query  →  EXPONENTIAL SPEEDUP           ║
║                                                                  ║
║  4. WHY DOES IT WORK?                                            ║
║     Quantum interference in the second Hadamard layer:           ║
║     • Constant oracle: all (−1)^f(x) are equal                  ║
║       → amplitudes add constructively at |0...0⟩               ║
║     • Balanced oracle: half +1, half −1 phases                  ║
║       → amplitudes cancel to 0 at |0...0⟩                      ║
║                                                                  ║
║  5. HISTORICAL SIGNIFICANCE                                      ║
║     • First algorithm to provably outperform all classical ones ║
║     • Introduced key techniques: superposition, oracle,         ║
║       phase kickback, interference — used in ALL major          ║
║       algorithms: Shor, Grover, Simon, HHL, QPE, etc.           ║
╚══════════════════════════════════════════════════════════════════╝
"""
print(summary)

print("Qiskit version used :", qiskit.__version__)
print("Simulator           :", SIMULATOR.name)
print()
print("Files saved in /tmp/:")
import os
for f in ['oracles.png', 'kickback.png', 'deutsch.png',
           'dj_histograms.png', 'dj_statevector.png',
           'dj_complexity.png', 'bloch_final.png', 'dj_interactive.png']:
    path = f'/tmp/{f}'
    size = os.path.getsize(path) if os.path.exists(path) else 0
    status = f"✅ {size//1024} KB" if size > 0 else "❌ not found"
    print(f"  {f:<30} {status}")